# 03 - PyTorch models, logits and genuine calendar sequences

Qidian (Martin) Xing. Independent extension with disclosed Codex assistance. The historical period was previously inspected. Source data and row-level predictions are not redistributed.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))
from distress_ml.common import ROOT, PUBLIC, ART
pd.set_option("display.max_columns", 12)
print("Independent extension: saved real-data evidence; no training is triggered.")


Independent extension: saved real-data evidence; no training is triggered.


## Question
Does a more complex network justify its cost, and does history matter independently of architecture?

## Logic
MLP, Residual MLP and FT-Transformer all consume tabular features. Feature attention is not temporal recurrence. GRU consumes actual calendar slots, with masks, plus flattened-lag controls on the same targets.

In [2]:
dev = pd.read_csv(PUBLIC / "development_summary.csv")
display(dev[dev.family.isin(["mlp", "residual_mlp", "ft_transformer", "gru"])][["family", "feature_set", "stage", "macro_year_AP", "mean_fit_seconds"]])

,family,feature_set,stage,macro_year_AP,mean_fit_seconds
2,residual_mlp,S1,controlled_ablation,0.273532,25.324360
6,residual_mlp,S1,controlled_ablation,0.240091,28.423375
12,residual_mlp,S1,bounded_search,0.233008,21.552346
19,residual_mlp,S1,controlled_ablation,0.208027,10.419640
20,residual_mlp,S1,core_recipe,0.205995,9.868024
21,residual_mlp,S1,controlled_ablation,0.204991,29.456026
22,residual_mlp,S0,core_recipe,0.197154,9.347462
24,mlp,S1,core_recipe,0.192435,6.911598
25,residual_mlp,S2,feature_ablation,0.191243,14.348470
26,residual_mlp,S1,controlled_ablation,0.190372,5.483325


`BCEWithLogitsLoss` consumes logits directly. AdamW updates train-mode parameters; scoring uses eval/inference mode. Stopping uses an inner tail year, followed by a new full-window fixed-length refit. The saved artifact below is the actual formal scoring model, not a new explanatory retrain.

In [3]:
freeze = json.loads((ROOT / "configs/final_freeze.json").read_text())
dl = next(m for m in freeze["models"] if "DL" in m["representative"] and m["seed"] == 42)
from distress_ml.train import load_pipeline
if (ROOT / dl["path"]).exists():
    model = load_pipeline(ROOT / dl["path"])
    print("Formal model family:", model.cfg["family"])
    print("Parameter count:", model.metadata["parameters"])
    print("Selected full-refit epochs:", model.metadata["epochs"])
    print("Eval mode:", not model.model.training)
    assert not model.model.training
else:
    print("Local model artifacts excluded; aggregate architecture and cost tables are available.")

Formal model family: residual_mlp
Parameter count: 36801
Selected full-refit epochs: 18
Eval mode: True


In [4]:
display(dev[dev.stage.isin(["sequence_control", "sequence_extension"])][["family", "feature_set", "macro_year_AP", "mean_fit_seconds"]])
print("Sequence implementation: sequences.py; feature tensors: [batch, calendar_time, feature].")

,family,feature_set,macro_year_AP,mean_fit_seconds
18,xgboost,LAGS,0.221405,4.683446
34,gru,LAGS,0.170498,13.140712
35,mlp,LAGS,0.170225,9.077432


Sequence implementation: sequences.py; feature tensors: [batch, calendar_time, feature].


Changing seeds does not create independent test data. Cross-device/version bit equality is not promised. The synthetic software tests check shape, finite loss/gradients, one update and checkpoint reload; they do not supply research performance.